In [1]:
import sys, os, warnings, subprocess
subprocess.run(["pip", "install", ".", "--quiet"], check=True,
               cwd="/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales")
warnings.simplefilter("ignore", UserWarning)

import numpy as np
import scipy as sp
import pandas as pd
import nibabel as nib
import pygam

import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 8})
plt.rcParams["svg.fonttype"] = "none"
import matplotlib.colors as mcolors
import seaborn as sns

sys.path.extend([r'/mnt/storage_ssd_raid/research_projects/snaplab_tools'])
sys.path.extend([r'/mnt/storage_ssd_raid/research_projects/nctpy/src'])

from src.notebook import load_notebook_data
from src.timescales import (create_ints_dataframe_custom_grouping, convert_data_dict,
                             average_data_dict_converted, compute_deflections)
from src.plotting import create_sydnor_sa_colormap
from src.gams import (fit_all_panels, compute_partial_r2_age, apply_region_mask,
                      get_series_colors, get_draw_order, draw_gam_figure,
                      plot_wholebrain_gam, plot_deriv_sa_correlation)

from snaplab_tools.plotting.plotting import plot_correlation, plot_brain_surface_data
from snaplab_tools.plotting.utils import get_my_colors

  DEPRECATION: Building 'src' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'src'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [2]:
my_colors = get_my_colors(cat_trio=False, as_list=True)

# Load data

In [3]:
dataset = 'HCP-D'
# dataset = 'HBN'
# dataset = 'PNC'
which_modality = 'FMRI'
# output_type = 'timescale_zero_crossing'
output_type = 'timescale_auc_zero_crossing'

# output_type = output_type + '_lp-0.01hz'
output_type = output_type + '_bp-0.01-0.08hz'
# output_type = output_type + '_bp-0.08-0.15hz'

notebook_data = load_notebook_data(dataset=dataset, which_modality=which_modality,
                                   output_type=output_type, use_resids=True)

Loading data for notebooks...
Initialized DataLoader for dataset: HCP-D
  Base directory:        /mnt/lindenmp_bignas/lindenmp/research_data/HCP_D
  Raw data directory:    /mnt/lindenmp_bignas/lindenmp/research_data/HCP_D/processed
  Derivatives directory: /mnt/lindenmp_bignas/lindenmp/research_data/HCP_D/derivatives
  Output directory:      /mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales
  Behavioral data:       565 subjects

	Initialized Timescales for dataset: HCP-D
	Output directory: /mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales
Loaded 2760 scans from scan_registry_HCP-D.csv


Loading metric_tsnr: 100%|██████████| 2760/2760 [00:00<00:00, 7540.70it/s]


2760
timescale_auc_zero_crossing_bp-0.01-0.08hz


Loading timescale_auc_zero_crossing_bp-0.01-0.08hz: 100%|██████████| 2760/2760 [00:00<00:00, 8228.27it/s]


2760


Loading timescale_auc_zero_crossing_bp-0.01-0.08hz_resids: 100%|██████████| 2760/2760 [00:00<00:00, 8385.39it/s]

2760
Number of subjects: 552


In [4]:
%matplotlib inline
plt.rcParams.update({
    'font.size':            8,
    'figure.dpi':           300,
    'svg.fonttype':         "none",
    'font.family':          'sans-serif',
    'font.sans-serif':      ['Roboto'],
    'axes.spines.top':      True,
    'axes.spines.right':    True
})

nas_scratch_dir = f'/mnt/lindenmp_nas/lindenmp/work/figure_scratch/intrinsic_neural_timescales/{dataset}'
if not os.path.exists(nas_scratch_dir):
    os.mkdir(nas_scratch_dir)

In [5]:
# Unpack notebook outputs
timescales_processor    = notebook_data['timescales_processor']
data_dict               = notebook_data['data_dict']
data_dict_tsnr          = notebook_data['data_dict_tsnr']
results_dir             = notebook_data['results_dir']
distance_matrix         = notebook_data['supporting_data']['distance_matrix']
gene_expression_nanmask = notebook_data['supporting_data']['gene_expression_nanmask']
yeo_systems             = notebook_data['supporting_data']['yeo_systems']

# Parcellation and SA axis
if dataset == 'HCP-D':
    parcellation = 'schaefer_400-7'
    sa_axis_file = os.path.join(timescales_processor.output_dir, 'data', 'supporting_data',
                                'schaefer400-7_sa-axis.npy')
elif dataset in ('PNC', 'NKI', 'HBN'):
    parcellation = 'schaefer_400-17'
    sa_axis_file = os.path.join(timescales_processor.output_dir, 'data', 'supporting_data',
                                'schaefer400-17_sa-axis.npy')
sa_axis = sp.stats.rankdata(np.load(sa_axis_file))


In [6]:
timescales_processor.scan_registry[:5]

[ScanInfo(subject_id='HCD2939275V1MR', task='rfMRIREST1PA', space='fsLR', seg='Schaefer4007', modality='MRI3T', tr=0.8, session='01', stat='mean', run=None, desc=None, den=None, rbc=False),
 ScanInfo(subject_id='HCD2939275V1MR', task='rfMRIREST2PA', space='fsLR', seg='Schaefer4007', modality='MRI3T', tr=0.8, session='01', stat='mean', run=None, desc=None, den=None, rbc=False),
 ScanInfo(subject_id='HCD2939275V1MR', task='tfMRICARITPA', space='fsLR', seg='Schaefer4007', modality='MRI3T', tr=0.8, session='01', stat='mean', run=None, desc=None, den=None, rbc=False),
 ScanInfo(subject_id='HCD2939275V1MR', task='tfMRIEMOTIONPA', space='fsLR', seg='Schaefer4007', modality='MRI3T', tr=0.8, session='01', stat='mean', run=None, desc=None, den=None, rbc=False),
 ScanInfo(subject_id='HCD2939275V1MR', task='tfMRIGUESSINGPA', space='fsLR', seg='Schaefer4007', modality='MRI3T', tr=0.8, session='01', stat='mean', run=None, desc=None, den=None, rbc=False)]

In [7]:
timescales_processor.match_registry_to_behavior()
df_behavior = timescales_processor.df_behavior_matched.copy()
subject_ids = list(dict.fromkeys(scan.subject_id for scan in timescales_processor.scan_registry))
n_subs      = len(subject_ids)
print(f'Number of subjects: {n_subs}')
df_behavior.head()


Number of subjects: 552


,age,sex,meanFD
subject_id,,,
HCD2939275V1MR,16.250000,0.0,0.097079
HCD0051825V1MR,18.166667,0.0,0.108911
HCD1168245V1MR,18.500000,1.0,0.157230
HCD1057337V1MR,19.416667,1.0,0.081428
HCD1701338V1MR,20.833333,1.0,0.108502


# Data for figures

In [8]:
data_dict_converted = convert_data_dict(data_dict, timescales_processor.scan_registry)
data_dict_centiles  = convert_data_dict(data_dict, timescales_processor.scan_registry,
                                        brain_map=sa_axis)

average_all_rest    = True
data_dict_converted = average_data_dict_converted(data_dict_converted,
                                                   average_all_rest=average_all_rest)
data_dict_centiles  = average_data_dict_converted(data_dict_centiles,
                                                   average_all_rest=average_all_rest)
df_tsnr = create_ints_dataframe_custom_grouping(data_dict_tsnr,
                                                 average_all_rest=average_all_rest)

if dataset == 'HCP-D':
    rest_scan = 'task-Rest'
    task_scan = 'tfMRICARITPA'
elif dataset == 'PNC':
    rest_scan = 'rest'
    task_scan = 'frac2back'
elif dataset == 'Penn_LEAD':
    rest_scan = 'rest'
    task_scan = 'nback'
elif dataset == 'HBN':
    rest_scan = 'rest'
    task_scan = None


rfMRIREST1PA (552, 400)
rfMRIREST2PA (552, 400)
tfMRICARITPA (552, 400)
tfMRIEMOTIONPA (552, 400)
tfMRIGUESSINGPA (552, 400)
rfMRIREST1PA (552, 10)
rfMRIREST2PA (552, 10)
tfMRICARITPA (552, 10)
tfMRIEMOTIONPA (552, 10)
tfMRIGUESSINGPA (552, 10)


# Figure 4: Age effects, GAMs

In [9]:
# mode: "decile"     — (n_subjects, n_deciles), SA-binned
#       "region"     — (n_subjects, n_regions), per-region
#       "wholebrain" — (n_subjects, 1), mean over all (non-masked) regions
mode    = 'decile'
show_ci = False
demean  = False

# Get region-level data (used as source for all modes)
rest_full = data_dict_converted[rest_scan]
try:
    task_full = data_dict_converted[task_scan]
except:
    task_full = data_dict_converted[rest_scan]

deflection_full = np.zeros(task_full.shape)
for i in np.arange(n_subs):
    deflection_full[i, :] = compute_deflections(rest_full[i, :], task_full[i, :],
                                                nuisance_regression=False,
                                                standardize=False, absolute=True)
age = df_behavior['age'].values

if dataset == 'PNC':
    q      = 0.8
    meanFD = df_behavior[[f'meanFD_{rest_scan}', f'meanFD_{task_scan}']].mean(axis=1)
    mask   = meanFD <= np.quantile(meanFD, q=q)
    rest_full, task_full, deflection_full, age = (
        rest_full[mask], task_full[mask], deflection_full[mask], age[mask])

if mode == 'decile':
    rest       = data_dict_centiles[rest_scan]
    try:
        task = data_dict_centiles[task_scan]
    except:
        task = data_dict_centiles[rest_scan]
    deflection = np.zeros(task.shape)
    for i in np.arange(n_subs if len(age) == n_subs else len(age)):
        deflection[i, :] = compute_deflections(rest[i, :], task[i, :],
                                               nuisance_regression=False,
                                               standardize=False, absolute=True)
    region_mask   = None
    sa_axis_local = None   # coloring handled by SA-binned centile index
elif mode == 'region':
    region_mask = (df_tsnr[task_scan] < np.quantile(df_tsnr[task_scan], q=0.1)).values
    rest, task, deflection, sa_axis_local = apply_region_mask(
        region_mask, rest_full, task_full, deflection_full, sa_axis=sa_axis)
elif mode == 'wholebrain':
    region_mask = (df_tsnr[task_scan] < np.quantile(df_tsnr[task_scan], q=0.1)).values
    rest_m, task_m, defl_m, _ = apply_region_mask(
        region_mask, rest_full, task_full, deflection_full, sa_axis=None)
    # average over all remaining regions → (n_subjects, 1)
    rest       = rest_m.mean(axis=1, keepdims=True)
    task       = task_m.mean(axis=1, keepdims=True)
    deflection = defl_m.mean(axis=1, keepdims=True)
    sa_axis_local = None

n_series = rest.shape[1]
cmap     = create_sydnor_sa_colormap()
if mode != 'wholebrain':
    series_colors = get_series_colors(mode, n_series, cmap, sa_axis=sa_axis_local)
    draw_order    = get_draw_order(mode, n_series, sa_axis=sa_axis_local)
print(f'mode={mode}  rest {rest.shape}  task {task.shape}  deflection {deflection.shape}  age {age.shape}')


mode=decile  rest (552, 10)  task (552, 10)  deflection (552, 10)  age (552,)


In [10]:
if dataset == 'HCP-D':
    nuisance_cols = ['sex', 'meanFD']
    nuisance      = df_behavior[nuisance_cols].values
    print(f'Nuisance covariates: {nuisance_cols}, shape {nuisance.shape}')
else:
    nuisance = None

age_grid = np.linspace(age.min(), age.max(), 200)
results  = fit_all_panels(age, rest, task, deflection, age_grid,
                          n_splines=4, lam=0.1, demean=demean, show_ci=show_ci,
                          nuisance=nuisance)


Nuisance covariates: ['sex', 'meanFD'], shape (552, 2)


In [11]:
if mode != 'wholebrain':
    # panels_cfg = [
    #     ('rest',       'Rest INT',       'Rest INT'),
    #     ('task',       'Task INT',       'Task INT'),
    #     ('deflection', 'INT Deflections', 'INT Deflections'),
    # ]
    panels_cfg = [
        ('deflection', 'INT Deflections', 'INT Deflections'),
    ]
    fig1 = draw_gam_figure(results, series_colors, draw_order, panels_cfg, 'pred',
                           age_grid, mode, n_series, cmap, demean, show_ci)
    fig1.savefig(os.path.join(nas_scratch_dir, f'gams_age_saaxis_{mode}.png'), bbox_inches='tight', dpi=1000, transparent=True)

    fig2 = draw_gam_figure(results, series_colors, draw_order, panels_cfg, 'deriv',
                           age_grid, mode, n_series, cmap, demean, show_ci)
    fig2.savefig(os.path.join(nas_scratch_dir, f'gams_age_saaxis_dervis_{mode}.png'), bbox_inches='tight', dpi=1000, transparent=True)


In [12]:
if mode == 'wholebrain':
    panels_cfg = [
        ('rest',       'Rest INT',       'Whole-brain mean INT'),
        ('task',       'Task INT',       'Whole-brain mean INT'),
        ('deflection', 'Deflection INT', 'Whole-brain mean deflection'),
    ]
    condition_data = {'rest': rest, 'task': task, 'deflection': deflection}

    # GAM smooth with subject scatter
    fig_wb = plot_wholebrain_gam(results, panels_cfg, 'pred', age, demean,
                                 show_ci=show_ci, condition_data=condition_data)

    # Temporal derivative of the GAM smooth
    # fig_wb_deriv = plot_wholebrain_gam(results, panels_cfg, 'deriv', age, demean,
    #                                    show_ci=False)


In [13]:
if mode == 'region':
    fig = plot_deriv_sa_correlation(results, sa_axis_local, method='pearson')
    fig.savefig(os.path.join(nas_scratch_dir, f'gams_derivative_corr_age-saaxis_{mode}.png'), bbox_inches='tight', dpi=1000, transparent=True)

    # Replicate the rho computation from plot_deriv_sa_correlation to identify key age points
    deriv  = results['deflection']['deriv']
    n_grid = deriv.shape[1]
    rhos   = np.full(n_grid, np.nan)
    for t in range(n_grid):
        col  = deriv[:, t]
        mask = ~np.isnan(col)
        if mask.sum() > 2:
            rhos[t] = sp.stats.pearsonr(sa_axis_local[mask], col[mask]).statistic

    valid   = np.where(~np.isnan(rhos))[0]
    t_young = int(valid[0])
    t_old   = int(valid[-1])
    t_peak  = int(np.nanargmax(np.abs(rhos)))

    yeo_masked = np.asarray(yeo_systems)[~region_mask]

    for t_idx, title in [(t_young, f'Youngest age ({age_grid[t_young]:.1f} yrs)'),
                         (t_old,   f'Oldest age ({age_grid[t_old]:.1f} yrs)'),
                         (t_peak,  f'Peak |r| ({age_grid[t_peak]:.1f} yrs)')]:
        col  = deriv[:, t_idx]
        mask = ~np.isnan(col)
        fig_c, ax_c = plt.subplots(figsize=(5.5, 5))
        plot_correlation(sa_axis_local[mask], col[mask], ax_c,
                         x_label='SA Axis', y_label='GAM derivative (deflection INT)',
                         title=title,
                         data_group=yeo_masked[mask],
                         show_stats=True, fontsize=11, method='pearson',
                         size=35, cbar_label='Yeo Systems',
                         auto_polynomial=True, models_to_test=[1],
                         model_selection_metric='variance_explained')
        plt.show()


In [14]:
if mode == 'region':
    n_parcels = 400
    _excluded = region_mask

    for condition_name, condition_data in [('rest', rest), ('task', task),
                                           ('deflection', deflection)]:
        gams = results[condition_name]['gams']
        pr2  = compute_partial_r2_age(condition_data, age, gams,
                                      nuisance=nuisance, demean=demean)

        full_pr2             = np.full(n_parcels, np.nan)
        full_pr2[~_excluded] = pr2

        # fig, axes = plot_brain_surface_data(
        #     data_vector    = full_pr2,
        #     data_mask      = _excluded,
        #     parcellation   = parcellation,
        #     cmap           = 'coolwarm',
        #     symmetric_cbar = False,
        #     title          = f'Partial R² (age) — {condition_name}',
        #     figsize        = (8, 6),
        #     fontsize       = 8,
        #     cbar_label     = 'Partial R² (age)',
        # )
        # plt.show()

        fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5.5, 5))
        plot_correlation(sa_axis[~_excluded], full_pr2[~_excluded], ax,
                         data_group=np.asarray(yeo_systems)[~_excluded],
                         show_stats=True, fontsize=11, method='pearson',
                         size=35, cbar_label='Yeo Systems',
                         y_label=condition_name, x_label='SA Axis',
                         auto_polynomial=True, models_to_test=[1],
                         model_selection_metric='variance_explained')
        plt.show()
